# PySpark Tutorial — US Housing Market Data

A hands-on tour of Apache Spark running **locally on Windows**, using a realistic
US housing-sales dataset (50,000 sales across 30 cities, 2021–2024).

**Sections**

| # | Topic | # | Topic |
|---|-------|---|-------|
| 0 | Environment & SparkSession | 7 | Joins |
| 1 | Loading data & schemas | 8 | Window functions |
| 2 | First look at the data | 9 | Spark SQL |
| 3 | Select, filter, transform | 10 | UDFs & pandas UDFs |
| 4 | Column functions (string / date / math) | 11 | Under the hood: plans, caching, partitions |
| 5 | Nulls & data cleaning | 12 | Writing files (Parquet / CSV / JSON) |
| 6 | Aggregations & pivot | 13 | SQLite round-trip (write → read → update → save back) |

> Run cells top to bottom the first time. After that, each section mostly stands alone.
> The dataset comes from `generate_data.py` (already run) — see `data/housing_sales.csv`.

## 0 · Environment & SparkSession

Spark is a JVM application; PySpark drives it from Python. On this machine everything
Spark needs lives **inside the project folder** so nothing depends on system config:

- `jdk/` — portable Temurin JDK 17 (`JAVA_HOME`)
- `hadoop/bin/` — `winutils.exe` + `hadoop.dll`, required on Windows for file writes (`HADOOP_HOME`)
- `.venv/` — Python 3.11 with `pyspark==3.5.7`
- `jars/sqlite-jdbc-*.jar` — JDBC driver for the SQLite section

`PYSPARK_PYTHON` is pinned to the venv interpreter — otherwise Spark spawns workers with
whatever `python3` is on PATH (here: Python 3.14, which PySpark 3.5 does not support).

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd()                       # d:/1sde/0databricks/2.spark
JDK = next((PROJECT / "jdk").glob("jdk-*"))
SQLITE_JAR = next((PROJECT / "jars").glob("sqlite-jdbc-*.jar"))

os.environ["JAVA_HOME"] = str(JDK)
os.environ["HADOOP_HOME"] = str(PROJECT / "hadoop")
os.environ["PATH"] = os.pathsep.join([str(JDK / "bin"), str(PROJECT / "hadoop" / "bin"), os.environ["PATH"]])
os.environ["PYSPARK_PYTHON"] = sys.executable          # workers must use THIS python
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("JAVA_HOME  =", os.environ["JAVA_HOME"])
print("python     =", sys.executable)

JAVA_HOME  = D:\1sde\0databricks\2.spark\jdk\jdk-17.0.20+8
python     = D:\1sde\0databricks\2.spark\.venv\Scripts\python.exe


The `SparkSession` is the entry point to everything. Worth knowing:

- `master("local[*]")` — run locally using all CPU cores (each core ≈ one task at a time)
- `spark.sql.shuffle.partitions` — defaults to **200**, absurd for a laptop; 8 keeps shuffles fast
- `spark.jars` — extra JVM jars (our SQLite JDBC driver)
- While the session is alive, the **Spark UI** is at <http://localhost:4040> — open it and watch
  jobs/stages/tasks appear as you run cells. It's the single best tool for understanding Spark.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("housing-tutorial")
    .config("spark.jars", str(SQLITE_JAR))
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")   # silence INFO spam
spark

## 1 · Loading data & schemas

Two ways to get a schema:

1. **`inferSchema=True`** — Spark reads the file *twice* (once to guess types). Fine for exploration.
2. **Explicit `StructType`** — one pass, no guessing, and bad rows surface immediately. Use in real pipelines.

Spark reads are **lazy**: `spark.read...` builds a plan; nothing touches the file until an
*action* (`count`, `show`, `write`, `collect`) runs.

In [ ]:
# 1a — quick & lazy: let Spark infer types
raw = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .csv("data/housing_sales.csv"))
raw.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: integer (nullable = true)
 |-- lot_sqft: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- year_renovated: integer (nullable = true)
 |-- garage_spaces: integer (nullable = true)
 |-- hoa_monthly: integer (nullable = true)
 |-- sale_date: date (nullable = true)



In [ ]:
# 1b — production style: explicit schema (faster, stricter, self-documenting)
schema = T.StructType([
    T.StructField("sale_id",        T.IntegerType(), False),
    T.StructField("address",        T.StringType(),  True),
    T.StructField("city",           T.StringType(),  True),
    T.StructField("state",          T.StringType(),  True),
    T.StructField("property_type",  T.StringType(),  True),
    T.StructField("price",          T.LongType(),    True),
    T.StructField("bedrooms",       T.IntegerType(), True),
    T.StructField("bathrooms",      T.DoubleType(),  True),
    T.StructField("sqft",           T.IntegerType(), True),
    T.StructField("lot_sqft",       T.IntegerType(), True),
    T.StructField("year_built",     T.IntegerType(), True),
    T.StructField("year_renovated", T.IntegerType(), True),   # has nulls
    T.StructField("garage_spaces",  T.IntegerType(), True),   # has nulls
    T.StructField("hoa_monthly",    T.IntegerType(), True),
    T.StructField("sale_date",      T.DateType(),    True),
])

sales = (spark.read
         .option("header", True)
         .schema(schema)
         .csv("data/housing_sales.csv"))

city_stats = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv("data/city_stats.csv"))

print(f"sales rows: {sales.count():,}   city_stats rows: {city_stats.count()}")

sales rows: 50,000   city_stats rows: 30


## 2 · First look at the data

The everyday inspection toolkit: `show`, `printSchema`, `columns` / `dtypes`,
`describe` / `summary`, and `toPandas` (only ever on *small* results — it pulls
everything into driver memory).

In [ ]:
sales.show(5, truncate=False)

+-------+---------------+--------------+-----+-------------+------+--------+---------+----+--------+----------+--------------+-------------+-----------+----------+
|sale_id|address        |city          |state|property_type|price |bedrooms|bathrooms|sqft|lot_sqft|year_built|year_renovated|garage_spaces|hoa_monthly|sale_date |
+-------+---------------+--------------+-----+-------------+------+--------+---------+----+--------+----------+--------------+-------------+-----------+----------+
|1      |3578 Main Ct   |Raleigh       |NC   |Single Family|582368|4       |2.5      |2647|9256    |2007      |2023          |1            |0          |2021-03-07|
|2      |12776 Hill Dr  |Raleigh       |NC   |Condo        |360680|3       |2.0      |1497|0       |1995      |NULL          |0            |494        |2022-07-24|
|3      |38030 Maple Ave|Charlotte     |NC   |Single Family|492195|1       |1.0      |1646|8787    |1999      |NULL          |0            |250        |2023-01-11|
|4      |60689 P

In [ ]:
print(sales.columns)
print()
print(sales.dtypes)

['sale_id', 'address', 'city', 'state', 'property_type', 'price', 'bedrooms', 'bathrooms', 'sqft', 'lot_sqft', 'year_built', 'year_renovated', 'garage_spaces', 'hoa_monthly', 'sale_date']

[('sale_id', 'int'), ('address', 'string'), ('city', 'string'), ('state', 'string'), ('property_type', 'string'), ('price', 'bigint'), ('bedrooms', 'int'), ('bathrooms', 'double'), ('sqft', 'int'), ('lot_sqft', 'int'), ('year_built', 'int'), ('year_renovated', 'int'), ('garage_spaces', 'int'), ('hoa_monthly', 'int'), ('sale_date', 'date')]


In [ ]:
# summary statistics for the numeric columns you care about
sales.select("price", "bedrooms", "bathrooms", "sqft", "year_built").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

+-------+-----------------+------------------+------------------+-----------------+------------------+
|summary|            price|          bedrooms|         bathrooms|             sqft|        year_built|
+-------+-----------------+------------------+------------------+-----------------+------------------+
|  count|            50000|             50000|             50000|            50000|             50000|
|   mean|      844113.7923|           3.28746|           2.47726|       2128.71568|        1990.49586|
| stddev|528203.2900288574|1.1489356594096083|0.9898396274395971|623.2379231339459|27.042204948512783|
|    min|            60000|                 1|               1.0|              300|              1900|
|    25%|           487948|                 3|               1.5|             1686|              1976|
|    50%|           686306|                 3|               2.5|             2112|              1997|
|    75%|          1031935|                 4|               3.0|        

In [ ]:
# .limit() + .toPandas() = nice notebook display for a small sample
sales.limit(5).toPandas()

,sale_id,address,city,state,property_type,price,bedrooms,bathrooms,sqft,lot_sqft,year_built,year_renovated,garage_spaces,hoa_monthly,sale_date
0,1,3578 Main Ct,Raleigh,NC,Single Family,582368,4,2.5,2647,9256,2007,2023.0,1,0,2021-03-07
1,2,12776 Hill Dr,Raleigh,NC,Condo,360680,3,2.0,1497,0,1995,NaN,0,494,2022-07-24
2,3,38030 Maple Ave,Charlotte,NC,Single Family,492195,1,1.0,1646,8787,1999,NaN,0,250,2023-01-11
3,4,60689 Park Dr,Salt Lake City,UT,Single Family,810144,3,2.5,2505,6618,2021,NaN,1,250,2024-08-20
4,5,27969 River Ln,Columbus,OH,Condo,360943,3,2.5,1707,0,2013,NaN,1,217,2022-10-09


## 3 · Select, filter, transform

The core DataFrame verbs. Everything here is a **transformation** — it returns a *new*
DataFrame (immutable, lazily evaluated) and never mutates the original.

Three equivalent ways to reference a column: `F.col("price")`, `sales.price`, or the
string `"price"` (where accepted). `F.col` is the most general — use it by default.

In [ ]:
# select + expressions + alias
(sales
 .select(
     "city", "state", "price", "sqft",
     (F.col("price") / F.col("sqft")).alias("price_per_sqft"),
 )
 .show(5))

+--------------+-----+------+----+------------------+
|          city|state| price|sqft|    price_per_sqft|
+--------------+-----+------+----+------------------+
|       Raleigh|   NC|582368|2647|220.01057801284472|
|       Raleigh|   NC|360680|1497|240.93520374081496|
|     Charlotte|   NC|492195|1646| 299.0249088699878|
|Salt Lake City|   UT|810144|2505|323.41077844311377|
|      Columbus|   OH|360943|1707|211.44874048037494|
+--------------+-----+------+----+------------------+
only showing top 5 rows



In [ ]:
# filter / where (they are synonyms) — combine with & | ~, parenthesize each condition!
tx_big = sales.where(
    (F.col("state") == "TX") &
    (F.col("bedrooms") >= 4) &
    (F.col("price") < 500_000)
)
tx_big.select("city", "price", "bedrooms", "sqft", "sale_date").show(5)
print("matches:", tx_big.count())

+-----------+------+--------+----+----------+
|       city| price|bedrooms|sqft| sale_date|
+-----------+------+--------+----+----------+
|San Antonio|494181|       5|2538|2024-03-29|
|    Houston|406632|       4|2094|2023-11-10|
|     Dallas|449855|       4|1824|2024-01-26|
|    Houston|495278|       4|2437|2022-08-06|
|    Houston|496659|       4|2734|2023-01-14|
+-----------+------+--------+----+----------+
only showing top 5 rows



matches: 711


In [ ]:
# withColumn — add or replace columns; when/otherwise = SQL CASE WHEN
enriched = (
    sales
    .withColumn("price_per_sqft", F.round(F.col("price") / F.col("sqft"), 2))
    .withColumn("age_years", F.lit(2025) - F.col("year_built"))
    .withColumn("price_band",
        F.when(F.col("price") < 300_000, "budget")
         .when(F.col("price") < 750_000, "mid")
         .when(F.col("price") < 1_500_000, "premium")
         .otherwise("luxury"))
    .withColumn("is_renovated", F.col("year_renovated").isNotNull())
    .withColumnRenamed("hoa_monthly", "hoa_fee")
)
enriched.select("city", "price", "price_per_sqft", "age_years", "price_band", "is_renovated").show(5)

+--------------+------+--------------+---------+----------+------------+
|          city| price|price_per_sqft|age_years|price_band|is_renovated|
+--------------+------+--------------+---------+----------+------------+
|       Raleigh|582368|        220.01|       18|       mid|        true|
|       Raleigh|360680|        240.94|       30|       mid|       false|
|     Charlotte|492195|        299.02|       26|       mid|       false|
|Salt Lake City|810144|        323.41|        4|   premium|       false|
|      Columbus|360943|        211.45|       12|       mid|       false|
+--------------+------+--------------+---------+----------+------------+
only showing top 5 rows



In [ ]:
# distinct, dropDuplicates, orderBy, cast, drop
print("property types:", [r[0] for r in enriched.select("property_type").distinct().collect()])

(enriched
 .select("city", "state", "price", "sqft")
 .orderBy(F.col("price").desc())          # most expensive sales
 .show(5))

# cast: change a column's type
enriched.select(F.col("price").cast("double").alias("price_dbl")).printSchema()

property types: ['Single Family', 'Condo', 'Townhouse', 'Multi-Family']


+-------------+-----+-------+----+
|         city|state|  price|sqft|
+-------------+-----+-------+----+
|San Francisco|   CA|5165819|3993|
|San Francisco|   CA|4646373|3996|
|San Francisco|   CA|4415927|3736|
|San Francisco|   CA|4412227|3578|
|San Francisco|   CA|4402741|3225|
+-------------+-----+-------+----+
only showing top 5 rows

root
 |-- price_dbl: double (nullable = true)



## 4 · Column functions: string, date, math

`pyspark.sql.functions` (imported as `F`) has ~400 built-ins. They run in the JVM —
**always prefer them over Python UDFs** (Section 10 shows why).

In [ ]:
# string functions
(sales
 .select(
     "address",
     F.upper("city").alias("city_upper"),
     F.concat_ws(", ", "city", "state").alias("location"),
     F.split(F.col("address"), " ").getItem(0).alias("street_no"),
     F.regexp_extract("address", r"(St|Ave|Dr|Ln|Blvd|Ct)$", 1).alias("street_type"),
     F.length("address").alias("addr_len"),
 )
 .show(5, truncate=False))

+---------------+--------------+------------------+---------+-----------+--------+
|address        |city_upper    |location          |street_no|street_type|addr_len|
+---------------+--------------+------------------+---------+-----------+--------+
|3578 Main Ct   |RALEIGH       |Raleigh, NC       |3578     |Ct         |12      |
|12776 Hill Dr  |RALEIGH       |Raleigh, NC       |12776    |Dr         |13      |
|38030 Maple Ave|CHARLOTTE     |Charlotte, NC     |38030    |Ave        |15      |
|60689 Park Dr  |SALT LAKE CITY|Salt Lake City, UT|60689    |Dr         |13      |
|27969 River Ln |COLUMBUS      |Columbus, OH      |27969    |Ln         |14      |
+---------------+--------------+------------------+---------+-----------+--------+
only showing top 5 rows



In [ ]:
# date functions — sale_date is already a DateType thanks to our schema
(sales
 .select(
     "sale_date",
     F.year("sale_date").alias("yr"),
     F.month("sale_date").alias("mo"),
     F.quarter("sale_date").alias("qtr"),
     F.date_format("sale_date", "yyyy-MM").alias("year_month"),
     F.dayofweek("sale_date").alias("dow"),
     F.datediff(F.current_date(), "sale_date").alias("days_ago"),
     F.add_months("sale_date", 6).alias("plus_6mo"),
 )
 .show(5))

+----------+----+---+---+----------+---+--------+----------+
| sale_date|  yr| mo|qtr|year_month|dow|days_ago|  plus_6mo|
+----------+----+---+---+----------+---+--------+----------+
|2021-03-07|2021|  3|  1|   2021-03|  1|    1989|2021-09-07|
|2022-07-24|2022|  7|  3|   2022-07|  1|    1485|2023-01-24|
|2023-01-11|2023|  1|  1|   2023-01|  4|    1314|2023-07-11|
|2024-08-20|2024|  8|  3|   2024-08|  3|     727|2025-02-20|
|2022-10-09|2022| 10|  4|   2022-10|  1|    1408|2023-04-09|
+----------+----+---+---+----------+---+--------+----------+
only showing top 5 rows



In [ ]:
# math / misc
(sales
 .select(
     "price",
     F.round(F.col("price") / 1000, 1).alias("price_k"),
     F.log10("price").alias("log10_price"),
     F.floor(F.col("sqft") / 500).alias("sqft_bucket"),
     F.greatest("bedrooms", "garage_spaces").alias("max_of_two"),
     F.hash("address").alias("addr_hash"),
 )
 .show(5))

+------+-------+-----------------+-----------+----------+-----------+
| price|price_k|      log10_price|sqft_bucket|max_of_two|  addr_hash|
+------+-------+-----------------+-----------+----------+-----------+
|582368|  582.4|5.765197503315227|          5|         4|-1349128354|
|360680|  360.7|5.557122061002994|          2|         3|-1237673936|
|492195|  492.2|5.692137197575969|          3|         1| 1699918508|
|810144|  810.1|5.908562219924431|          5|         3|-1429889597|
|360943|  360.9|5.557438623678513|          3|         3|  280251553|
+------+-------+-----------------+-----------+----------+-----------+
only showing top 5 rows



## 5 · Nulls & data cleaning

`garage_spaces` and `year_renovated` contain nulls on purpose. The toolkit:
`isNull` / `isNotNull`, `df.na.drop`, `df.na.fill`, `df.na.replace`, `F.coalesce`,
and `dropDuplicates`.

In [ ]:
# count nulls in every column — a one-liner worth memorizing
sales.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sales.columns
]).show()

+-------+-------+----+-----+-------------+-----+--------+---------+----+--------+----------+--------------+-------------+-----------+---------+
|sale_id|address|city|state|property_type|price|bedrooms|bathrooms|sqft|lot_sqft|year_built|year_renovated|garage_spaces|hoa_monthly|sale_date|
+-------+-------+----+-----+-------------+-----+--------+---------+----+--------+----------+--------------+-------------+-----------+---------+
|      0|      0|   0|    0|            0|    0|       0|        0|   0|       0|         0|         39849|         8328|          0|        0|
+-------+-------+----+-----+-------------+-----+--------+---------+----+--------+----------+--------------+-------------+-----------+---------+



In [ ]:
# fill nulls (per-column defaults), or drop rows that have any
cleaned = sales.na.fill({"garage_spaces": 0})            # unknown garage -> 0
cleaned = cleaned.withColumn(                            # renovated? else year_built
    "effective_year", F.coalesce("year_renovated", "year_built"))

strict = sales.na.drop(subset=["garage_spaces", "year_renovated"])  # keep only complete rows
print(f"original: {sales.count():,}   after na.drop: {strict.count():,}")

cleaned.select("year_built", "year_renovated", "effective_year", "garage_spaces").show(5)

original: 50,000   after na.drop: 8,488
+----------+--------------+--------------+-------------+
|year_built|year_renovated|effective_year|garage_spaces|
+----------+--------------+--------------+-------------+
|      2007|          2023|          2023|            1|
|      1995|          NULL|          1995|            0|
|      1999|          NULL|          1999|            0|
|      2021|          NULL|          2021|            1|
|      2013|          NULL|          2013|            1|
+----------+--------------+--------------+-------------+
only showing top 5 rows



## 6 · Aggregations & pivot

`groupBy(...).agg(...)` is the workhorse. Also here: multiple aggregates with aliases,
`approx_percentile` for medians (exact percentiles are expensive at scale), and `pivot`.

In [ ]:
# average / median price by state, most expensive first
(sales
 .groupBy("state")
 .agg(
     F.count("*").alias("n_sales"),
     F.round(F.avg("price")).alias("avg_price"),
     F.expr("approx_percentile(price, 0.5)").alias("median_price"),
     F.min("price").alias("min_price"),
     F.max("price").alias("max_price"),
 )
 .orderBy(F.col("median_price").desc())
 .show())

+-----+-------+---------+------------+---------+---------+
|state|n_sales|avg_price|median_price|min_price|max_price|
+-----+-------+---------+------------+---------+---------+
|   NY|   1679|1713025.0|     1671868|   294162|  4028070|
|   MA|   1677|1523694.0|     1486448|   303197|  3548047|
|   CA|   6587|1437077.0|     1309319|   116653|  5165819|
|   CO|   3338|1130190.0|     1072192|   222392|  3344876|
|   OR|   1714| 873732.0|      842014|   180973|  2024892|
|   WA|   3253| 900662.0|      780284|   134701|  2682481|
|   UT|   1655| 752906.0|      733920|    94372|  1674619|
|   FL|   5042| 768995.0|      712605|   139672|  2285046|
|   TN|   1684| 693360.0|      681504|   125051|  1586865|
|   GA|   1682| 622918.0|      605848|   133273|  1531453|
|   NV|   1567| 595635.0|      581048|   131252|  1335787|
|   NC|   3315| 598804.0|      580852|   100429|  1378839|
|   IL|   1699| 571524.0|      554739|   145152|  1379583|
|   MN|   1612| 548138.0|      539153|   111487|  123110

In [ ]:
# pivot: rows = state, columns = property_type, values = median price
(sales
 .groupBy("state")
 .pivot("property_type")
 .agg(F.expr("approx_percentile(price, 0.5)"))
 .orderBy("state")
 .show(10))

+-----+-------+------------+-------------+---------+
|state|  Condo|Multi-Family|Single Family|Townhouse|
+-----+-------+------------+-------------+---------+
|   AZ| 425325|      566158|       564371|   557171|
|   CA|1033267|     1379152|      1404817|  1361405|
|   CO| 847565|     1148731|      1124643|  1113825|
|   FL| 548608|      732687|       745790|   747255|
|   GA| 494088|      664325|       633911|   636868|
|   IL| 454701|      511852|       578116|   578690|
|   MA|1141291|     1526133|      1540932|  1564655|
|   MN| 427998|      538486|       567268|   562475|
|   NC| 456002|      628870|       614469|   611775|
|   NV| 452486|      637611|       609298|   592665|
+-----+-------+------------+-------------+---------+
only showing top 10 rows



In [ ]:
# yearly market trend — is the market appreciating? (it is, by design ~5%/yr)
(sales
 .groupBy(F.year("sale_date").alias("year"))
 .agg(
     F.count("*").alias("n_sales"),
     F.round(F.avg(F.col("price") / F.col("sqft")), 2).alias("avg_price_per_sqft"),
 )
 .orderBy("year")
 .show())

+----+-------+------------------+
|year|n_sales|avg_price_per_sqft|
+----+-------+------------------+
|2021|  12341|            369.04|
|2022|  12601|            387.72|
|2023|  12425|             403.4|
|2024|  12633|            424.08|
+----+-------+------------------+



## 7 · Joins

Join `sales` (50k rows) with `city_stats` (30 rows: median income, population).

- Join types: `inner` (default), `left`, `right`, `full`, `left_semi` (filter: keep left rows
  with a match), `left_anti` (keep left rows *without* a match).
- **`F.broadcast(small_df)`** ships the small table to every executor, skipping the shuffle —
  the #1 join optimization when one side is small. (Spark usually auto-broadcasts under 10MB,
  but being explicit documents intent.)

In [ ]:
joined = sales.join(
    F.broadcast(city_stats),
    on=["city", "state"],      # join on both to be safe (city names repeat across states)
    how="left",
)

# affordability: price as a multiple of local median household income
(joined
 .withColumn("income_multiple", F.round(F.col("price") / F.col("median_household_income"), 1))
 .groupBy("city")
 .agg(
     F.round(F.avg("income_multiple"), 1).alias("avg_income_multiple"),
     F.first("median_household_income").alias("median_income"),
 )
 .orderBy(F.col("avg_income_multiple").desc())
 .show(10))

+-------------+-------------------+-------------+
|         city|avg_income_multiple|median_income|
+-------------+-------------------+-------------+
|     New York|               22.5|        76000|
|        Miami|               19.2|        54000|
|  Los Angeles|               17.9|        76000|
|       Boston|               17.1|        89000|
|San Francisco|               17.0|       126000|
|    San Diego|               15.8|        89000|
|      Boulder|               14.2|        94000|
|      Seattle|               11.2|       110000|
|     Portland|               11.1|        79000|
|       Denver|               10.9|        85000|
+-------------+-------------------+-------------+
only showing top 10 rows



In [ ]:
# anti join: cities present in city_stats but with no sales in our data (should be none)
city_stats.join(sales, on=["city", "state"], how="left_anti").show()

# semi join: sales rows whose city appears in a filtered lookup (acts as a filter, adds no columns)
rich_cities = city_stats.where(F.col("median_household_income") > 100_000)
sales.join(rich_cities, on=["city", "state"], how="left_semi").groupBy("city").count().show()

+----+-----+-----------------------+----------+
|city|state|median_household_income|population|
+----+-----+-----------------------+----------+
+----+-----+-----------------------+----------+



+-------------+-----+
|         city|count|
+-------------+-----+
|San Francisco| 1669|
|      Seattle| 1683|
+-------------+-----+



## 8 · Window functions

Aggregates *without collapsing rows* — every row gets a value computed over its "window"
(partition + ordering + optional frame). The big three use cases:

1. **Ranking** — top-N per group (`row_number`, `rank`, `dense_rank`)
2. **Offsets** — compare to previous/next row (`lag`, `lead`)
3. **Running / moving aggregates** — `rowsBetween` frames

In [ ]:
from pyspark.sql import Window

# 1) top-3 most expensive sales per city
w_rank = Window.partitionBy("city").orderBy(F.col("price").desc())

(sales
 .withColumn("rn", F.row_number().over(w_rank))
 .where(F.col("rn") <= 3)
 .select("city", "rn", "price", "bedrooms", "sqft", "sale_date")
 .orderBy("city", "rn")
 .show(9))

+-------+---+-------+--------+----+----------+
|   city| rn|  price|bedrooms|sqft| sale_date|
+-------+---+-------+--------+----+----------+
|Atlanta|  1|1531453|       6|4143|2022-05-12|
|Atlanta|  2|1450768|       6|3582|2024-06-24|
|Atlanta|  3|1347242|       6|3657|2023-02-21|
| Austin|  1|1611821|       6|3787|2024-09-27|
| Austin|  2|1533930|       6|3591|2024-09-13|
| Austin|  3|1491131|       6|3841|2024-02-06|
| Boston|  1|3548047|       6|3685|2024-06-30|
| Boston|  2|3526843|       6|3777|2024-05-21|
| Boston|  3|3368602|       5|3574|2024-12-22|
+-------+---+-------+--------+----+----------+
only showing top 9 rows



In [ ]:
# 2) lag: month-over-month change in median price per sqft (whole market)
monthly = (sales
           .groupBy(F.date_format("sale_date", "yyyy-MM").alias("ym"))
           .agg(F.expr("approx_percentile(price / sqft, 0.5)").alias("median_ppsf")))

w_time = Window.orderBy("ym")
(monthly
 .withColumn("prev", F.lag("median_ppsf").over(w_time))
 .withColumn("mom_pct", F.round(100 * (F.col("median_ppsf") - F.col("prev")) / F.col("prev"), 2))
 .orderBy("ym")
 .show(12))

+-------+------------------+------------------+-------+
|     ym|       median_ppsf|              prev|mom_pct|
+-------+------------------+------------------+-------+
|2021-01|288.22444444444443|              NULL|   NULL|
|2021-02| 294.5014895729891|288.22444444444443|   2.18|
|2021-03|294.61664190193164| 294.5014895729891|   0.04|
|2021-04| 288.4583635047067|294.61664190193164|  -2.09|
|2021-05|296.23396081444486| 288.4583635047067|    2.7|
|2021-06| 303.7246256239601|296.23396081444486|   2.53|
|2021-07|294.41466580142765| 303.7246256239601|  -3.07|
|2021-08| 292.2706633557999|294.41466580142765|  -0.73|
|2021-09| 289.9062059238364| 292.2706633557999|  -0.81|
|2021-10|296.50467749876907| 289.9062059238364|   2.28|
|2021-11|288.49679113185533|296.50467749876907|   -2.7|
|2021-12|302.89061689994816|288.49679113185533|   4.99|
+-------+------------------+------------------+-------+
only showing top 12 rows



In [ ]:
# 3) moving average: 3-month rolling mean of the monthly median
w_roll = Window.orderBy("ym").rowsBetween(-2, 0)   # this row + 2 preceding
(monthly
 .withColumn("rolling_3mo", F.round(F.avg("median_ppsf").over(w_roll), 2))
 .orderBy("ym")
 .show(12))

+-------+------------------+-----------+
|     ym|       median_ppsf|rolling_3mo|
+-------+------------------+-----------+
|2021-01|288.22444444444443|     288.22|
|2021-02| 294.5014895729891|     291.36|
|2021-03|294.61664190193164|     292.45|
|2021-04| 288.4583635047067|     292.53|
|2021-05|296.23396081444486|      293.1|
|2021-06| 303.7246256239601|     296.14|
|2021-07|294.41466580142765|     298.12|
|2021-08| 292.2706633557999|      296.8|
|2021-09| 289.9062059238364|      292.2|
|2021-10|296.50467749876907|     292.89|
|2021-11|288.49679113185533|     291.64|
|2021-12|302.89061689994816|     295.96|
+-------+------------------+-----------+
only showing top 12 rows



## 9 · Spark SQL

Any DataFrame can be registered as a **temp view** and queried with plain SQL.
SQL and the DataFrame API compile to the *same* plans — pick whichever reads better,
and mix freely (a SQL result is just another DataFrame).

In [ ]:
sales.createOrReplaceTempView("sales")
city_stats.createOrReplaceTempView("city_stats")

spark.sql("""
    SELECT s.state,
           count(*)                            AS n_sales,
           round(avg(s.price))                 AS avg_price,
           round(avg(s.price / s.sqft), 2)     AS avg_ppsf,
           max(c.median_household_income)      AS top_median_income
    FROM sales s
    JOIN city_stats c USING (city, state)
    WHERE s.property_type = 'Single Family'
    GROUP BY s.state
    ORDER BY avg_ppsf DESC
    LIMIT 10
""").show()

+-----+-------+---------+--------+-----------------+
|state|n_sales|avg_price|avg_ppsf|top_median_income|
+-----+-------+---------+--------+-----------------+
|   NY|   1022|1788234.0|  804.48|            76000|
|   MA|   1054|1587967.0|  706.92|            89000|
|   CA|   4059|1510316.0|  677.24|           126000|
|   CO|   2107|1186410.0|  530.47|            94000|
|   WA|   2013| 943238.0|  422.51|           110000|
|   OR|   1084| 914337.0|   411.9|            79000|
|   FL|   3151| 805220.0|  360.45|            62000|
|   UT|   1038| 785643.0|  350.34|            81000|
|   TN|   1020| 730472.0|  327.21|            70000|
|   GA|   1032| 650133.0|  296.25|            74000|
+-----+-------+---------+--------+-----------------+



In [ ]:
# SQL result -> DataFrame -> keep chaining with the API
expensive = spark.sql("SELECT * FROM sales WHERE price > 2000000")
expensive.groupBy("city").count().orderBy(F.col("count").desc()).show(5)

+-------------+-----+
|         city|count|
+-------------+-----+
|San Francisco|  924|
|     New York|  473|
|       Boston|  299|
|    San Diego|  180|
|  Los Angeles|  155|
+-------------+-----+
only showing top 5 rows



## 10 · UDFs & pandas UDFs

When no built-in exists, write a **UDF** — but know the cost: rows are serialized to a
Python worker process and back. A **pandas UDF** (vectorized, Arrow-based) processes whole
batches as `pandas.Series` and is usually 5–20× faster than a plain UDF.

Rule of thumb: built-in `F.*` ≫ pandas UDF ≫ plain UDF.

In [ ]:
# plain Python UDF: classify a listing (contrived — F.when would do this faster)
@F.udf(returnType=T.StringType())
def listing_label(beds, sqft):
    if beds is None or sqft is None:
        return "unknown"
    if beds <= 1 and sqft < 800:
        return "starter"
    if beds >= 5 or sqft > 4000:
        return "estate"
    return "family"

sales.select("bedrooms", "sqft", listing_label("bedrooms", "sqft").alias("label")) \
     .groupBy("label").count().show()

+-------+-----+
|  label|count|
+-------+-----+
| family|42576|
| estate| 6948|
|starter|  476|
+-------+-----+



In [ ]:
# pandas UDF: vectorized monthly mortgage payment (30yr fixed @ 6.5%, 20% down)
import pandas as pd

@F.pandas_udf(T.DoubleType())
def monthly_payment(price: pd.Series) -> pd.Series:
    principal = price * 0.80
    r = 0.065 / 12
    n = 360
    return (principal * r * (1 + r) ** n / ((1 + r) ** n - 1)).round(2)

(sales
 .withColumn("est_monthly_payment", monthly_payment("price"))
 .select("city", "price", "est_monthly_payment")
 .show(5))

+--------------+------+-------------------+
|          city| price|est_monthly_payment|
+--------------+------+-------------------+
|       Raleigh|582368|            2944.77|
|       Raleigh|360680|            1823.79|
|     Charlotte|492195|            2488.81|
|Salt Lake City|810144|            4096.53|
|      Columbus|360943|            1825.12|
+--------------+------+-------------------+
only showing top 5 rows



## 11 · Under the hood: plans, caching, partitions

- **`explain()`** — see the physical plan Spark actually runs (read bottom-up)
- **`cache()` / `persist()`** — keep a computed DataFrame in memory when you'll reuse it;
  it materializes on the *next action*, not at the call
- **`repartition(n)`** — full shuffle to n partitions (up or down); **`coalesce(n)`** — merge
  to fewer partitions *without* a shuffle (down only, cheap). Classic use: `coalesce(1)`
  before writing a single output file.

In [ ]:
# read the plan bottom-up: scan csv -> filter -> project -> broadcast join -> aggregate
(sales
 .join(F.broadcast(city_stats), ["city", "state"])
 .where(F.col("price") > 1_000_000)
 .groupBy("state").count()
).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[state#50], functions=[count(1)])
   +- Exchange hashpartitioning(state#50, 8), ENSURE_REQUIREMENTS, [plan_id=1720]
      +- HashAggregate(keys=[state#50], functions=[partial_count(1)])
         +- Project [state#50]
            +- BroadcastHashJoin [city#49, state#50], [city#94, state#95], Inner, BuildRight, false
               :- Project [city#49, state#50]
               :  +- Filter (((isnotnull(price#52L) AND (price#52L > 1000000)) AND isnotnull(city#49)) AND isnotnull(state#50))
               :     +- FileScan csv [city#49,state#50,price#52L] Batched: false, DataFilters: [isnotnull(price#52L), (price#52L > 1000000), isnotnull(city#49), isnotnull(state#50)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/1sde/0databricks/2.spark/data/housing_sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(price), GreaterThan(price,1000000), IsNotNull(city), IsNotNull(state)], ReadSchema: struct<

In [ ]:
import time

agg = joined.groupBy("city", "property_type").agg(F.avg("price").alias("avg_price"))

t0 = time.perf_counter(); agg.count(); cold = time.perf_counter() - t0
agg.cache()
agg.count()                                   # materializes the cache
t0 = time.perf_counter(); agg.count(); warm = time.perf_counter() - t0
print(f"uncached: {cold:.2f}s   cached: {warm:.2f}s")

uncached: 0.34s   cached: 0.17s


In [ ]:
print("partitions after read:   ", sales.rdd.getNumPartitions())
print("after repartition(16):   ", sales.repartition(16).rdd.getNumPartitions())
print("after coalesce(1):       ", sales.coalesce(1).rdd.getNumPartitions())
# repartition by COLUMN: co-locate each state's rows (useful before partitioned writes/joins)
by_state = sales.repartition("state")
print("after repartition(state):", by_state.rdd.getNumPartitions())

partitions after read:    2


after repartition(16):    16
after coalesce(1):        1


after repartition(state): 2


## 12 · Writing files: Parquet, CSV, JSON

**Parquet** is the default choice: columnar, compressed, schema included, and Spark can
skip whole files/row-groups when you filter (predicate pushdown). `partitionBy` writes
a folder per value — filters on that column then read only matching folders.

Note Spark writes a *directory* of part-files, not a single file — that's the
distributed model showing through.

In [ ]:
# parquet partitioned by state -> out/sales_parquet/state=TX/part-*.parquet ...
(enriched
 .write
 .mode("overwrite")
 .partitionBy("state")
 .parquet("out/sales_parquet"))

# csv + json flavors (coalesce(1) -> a single part-file inside the folder)
enriched.coalesce(1).write.mode("overwrite").option("header", True).csv("out/sales_csv")
enriched.limit(100).write.mode("overwrite").json("out/sales_json")

import pathlib
for p in sorted(pathlib.Path("out/sales_parquet").glob("state=*"))[:5]:
    print(p.name)

state=AZ
state=CA
state=CO
state=FL
state=GA


In [ ]:
# read it back — filtering on the partition column prunes folders (see PartitionFilters in the plan)
pq = spark.read.parquet("out/sales_parquet")
print("rows:", pq.count())
pq.where(F.col("state") == "WA").groupBy("city").count().show()

rows: 50000
+-------+-----+
|   city|count|
+-------+-----+
|Spokane| 1570|
|Seattle| 1683|
+-------+-----+



## 13 · SQLite round-trip: write → read → update → save back

Real pipelines constantly talk to relational databases over **JDBC**. SQLite gives us a
zero-install local RDBMS (`data/housing.db` — one file) using the driver jar we loaded
into the session at startup.

The dance:
1. **Write** a Spark DataFrame into SQLite
2. **Read** it back as a DataFrame
3. **Update** — two flavors:
   - *Spark-style*: JDBC has no `UPDATE` in Spark — you read, transform, **overwrite**
   - *SQL-style*: run a real `UPDATE` with Python's `sqlite3`, then re-read in Spark
4. **Save back** the final state

In [ ]:
DB_PATH = str(PROJECT / "data" / "housing.db")
JDBC_URL = f"jdbc:sqlite:{DB_PATH}"
JDBC_OPTS = {"driver": "org.sqlite.JDBC"}

# 13.1 WRITE — a compact listings table into SQLite
listings = enriched.select(
    "sale_id", "city", "state", "property_type",
    "price", "bedrooms", "bathrooms", "sqft", "price_band",
    F.col("sale_date").cast("string").alias("sale_date"),   # SQLite has no DATE type
)

(listings
 .coalesce(1)               # SQLite = single file, single writer: avoid parallel insert locks
 .write
 .mode("overwrite")
 .options(**JDBC_OPTS)
 .jdbc(JDBC_URL, table="listings"))

print("wrote listings ->", DB_PATH)

wrote listings -> D:\1sde\0databricks\2.spark\data\housing.db


In [ ]:
# 13.2 READ back from SQLite
db_listings = spark.read.options(**JDBC_OPTS).jdbc(JDBC_URL, table="listings")
db_listings.show(5)
print("rows in SQLite:", db_listings.count())

+-------+--------------+-----+-------------+------+--------+---------+----+----------+----------+
|sale_id|          city|state|property_type| price|bedrooms|bathrooms|sqft|price_band| sale_date|
+-------+--------------+-----+-------------+------+--------+---------+----+----------+----------+
|      1|       Raleigh|   NC|Single Family|582368|       4|      2.5|2647|       mid|2021-03-07|
|      2|       Raleigh|   NC|        Condo|360680|       3|      2.0|1497|       mid|2022-07-24|
|      3|     Charlotte|   NC|Single Family|492195|       1|      1.0|1646|       mid|2023-01-11|
|      4|Salt Lake City|   UT|Single Family|810144|       3|      2.5|2505|   premium|2024-08-20|
|      5|      Columbus|   OH|        Condo|360943|       3|      2.5|1707|       mid|2022-10-09|
+-------+--------------+-----+-------------+------+--------+---------+----+----------+----------+
only showing top 5 rows



rows in SQLite: 50000


### 13.3a · Update, Spark-style (read → transform → overwrite)

Spark's JDBC writer only appends or overwrites — no row updates. The standard pattern is
to read the table, transform it, and overwrite.

⚠️ **The self-overwrite trap**: JDBC reads are lazy. If you `overwrite` the very table you're
reading from, Spark drops the table *and then* tries to read it → data gone. Fix: **`cache()` +
`count()`** to materialize into memory first (or write to a staging table and swap).

In [ ]:
# scenario: a 3% market correction on every price
updated = (db_listings
           .withColumn("price", F.round(F.col("price") * 0.97).cast("long")))

updated.cache()
print("materialized rows:", updated.count())    # forces the read BEFORE the drop

(updated
 .coalesce(1)
 .write
 .mode("overwrite")
 .options(**JDBC_OPTS)
 .jdbc(JDBC_URL, table="listings"))

updated.unpersist()
spark.read.options(**JDBC_OPTS).jdbc(JDBC_URL, "listings") \
     .select(F.round(F.avg("price")).alias("avg_price_after_correction")).show()

materialized rows: 50000


+--------------------------+
|avg_price_after_correction|
+--------------------------+
|                    818790|
+--------------------------+



### 13.3b · Update, SQL-style (sqlite3), then re-read in Spark

Sometimes a plain SQL `UPDATE` is the right tool — a small correction shouldn't need a
distributed engine. Python's built-in `sqlite3` talks to the same file; Spark sees the
change on the next read. This is everyday interop: engines share the database, each doing
what it's best at.

In [ ]:
import sqlite3

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.execute(
        "UPDATE listings SET price_band = 'ultra-luxury' WHERE price >= 3000000")
    print("rows updated by sqlite3:", cur.rowcount)

# Spark re-reads and sees the SQL-side update
fresh = spark.read.options(**JDBC_OPTS).jdbc(JDBC_URL, "listings")
fresh.groupBy("price_band").count().orderBy(F.col("count").desc()).show()

rows updated by sqlite3: 191


+------------+-----+
|  price_band|count|
+------------+-----+
|         mid|25950|
|     premium|15916|
|      luxury| 5556|
|      budget| 2387|
|ultra-luxury|  191|
+------------+-----+



In [ ]:
# 13.4 SAVE BACK — final state out of SQLite into a Spark-side gold table (parquet)
final = spark.read.options(**JDBC_OPTS).jdbc(JDBC_URL, "listings")

(final
 .write
 .mode("overwrite")
 .partitionBy("price_band")
 .parquet("out/listings_gold"))

spark.read.parquet("out/listings_gold").groupBy("price_band") \
     .agg(F.count("*").alias("n"), F.round(F.avg("price")).alias("avg_price")) \
     .orderBy(F.col("avg_price").desc()).show()

+------------+-----+---------+
|  price_band|    n|avg_price|
+------------+-----+---------+
|ultra-luxury|  191|  3382337|
|      luxury| 5556|  1893106|
|     premium|15916|   995484|
|         mid|25950|   515110|
|      budget| 2387|   236349|
+------------+-----+---------+



## Wrap-up

You've covered the full daily-driver Spark surface: reads with explicit schemas,
transformations, built-in functions, cleaning, aggregations, joins, windows, SQL, UDFs,
plans/caching/partitioning, file formats, and a JDBC round-trip against a real database.

**Ideas to keep playing**
- Open <http://localhost:4040> and rerun Section 11 — find the broadcast join in the DAG
- Rewrite Section 8's rolling average in pure SQL (`OVER (ORDER BY ... ROWS BETWEEN ...)`)
- Swap the generated CSV for real data: Zillow ZHVI (`zillow.com/research/data`, direct CSV
  downloads) — it's *wide* (one column per month), so practice `melt`/`stack` to make it tidy
- Point Section 13 at PostgreSQL in Docker (`org.postgresql.Driver`) — only the URL changes

Stop the session when done (frees the JVM and port 4040):

In [ ]:
spark.stop()